In [22]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [23]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
228,You should not take what I am about to say lig...,positive
770,The above seemed a much more appropriate title...,negative
20,"""I haven't laughed this hard since granny got ...",negative
847,I have to say that there is one redeeming spec...,negative
170,I've watched this film a few times and I never...,negative


In [24]:
# data preprocessing
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\priti\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\priti\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [25]:
df = normalize_text(df)
df.head()

,review,sentiment
228,take say lightly seen many many film reviewed ...,positive
770,seemed much appropriate title suicidal underli...,negative
20,i laughed hard since granny got caught wringer...,negative
847,say one redeeming speck regard film firmly est...,negative
170,watched film time never really liked it fan te...,negative


In [26]:
df['sentiment'].value_counts()

sentiment
negative    261
positive    239
Name: count, dtype: int64

In [27]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [28]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
228,take say lightly seen many many film reviewed ...,1
770,seemed much appropriate title suicidal underli...,0
20,i laughed hard since granny got caught wringer...,0
847,say one redeeming speck regard film firmly est...,0
170,watched film time never really liked it fan te...,0


In [29]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [30]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [31]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [32]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/Pritish-23/MLOps-Capstone-Proj.mlflow')
dagshub.init(repo_owner='Pritish-23', repo_name='MLOps-Capstone-Proj', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


Initialized MLflow to track repo "Pritish-23/MLOps-Capstone-Proj"

2026-06-02 23:28:15,258 - INFO - Initialized MLflow to track repo "Pritish-23/MLOps-Capstone-Proj"


Repository Pritish-23/MLOps-Capstone-Proj initialized!

2026-06-02 23:28:15,262 - INFO - Repository Pritish-23/MLOps-Capstone-Proj initialized!


<Experiment: artifact_location='mlflow-artifacts:/8f49f489603042b0a30911591798ae9f', creation_time=1780423000299, experiment_id='0', last_update_time=1780423000299, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}>

In [33]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.20)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-06-02 23:28:15,600 - INFO - Starting MLflow run...
2026-06-02 23:28:16,131 - INFO - Logging preprocessing parameters...
2026-06-02 23:28:17,151 - INFO - Initializing Logistic Regression model...
2026-06-02 23:28:17,152 - INFO - Fitting the model...
2026-06-02 23:28:17,177 - INFO - Model training complete.
2026-06-02 23:28:17,178 - INFO - Logging model parameters...
2026-06-02 23:28:17,516 - INFO - Making predictions...
2026-06-02 23:28:17,519 - INFO - Calculating evaluation metrics...
2026-06-02 23:28:17,541 - INFO - Logging evaluation metrics...
2026-06-02 23:28:18,826 - INFO - Saving and logging the model...
c:\Users\priti\.conda\envs\atlas\lib\site-packages\_distutils_hack\__init__.py:11: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional